In [2]:
#imports

import pandas as pd
import numpy as np
import requests

In [3]:
#import data

data = pd.read_csv('Dataset4820.csv')
data

,Used Date,From,To,Airline,Scheduled Departure,SDEP,Departure,DEP,Scheduled Arrival,SARR,...,Market Share,OTP Index,weather__hourly__windspeedKmph,weather__hourly__weatherDesc__value,weather__hourly__precipMM,weather__hourly__humidity,weather__hourly__visibility,weather__hourly__pressure,weather__hourly__cloudcover,Category
0,31-01-2020,DEL,HYD,Air Asia,06:05,605,06:04,604,08:15,815,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,0
1,31-01-2020,DEL,HYD,Indigo,04:50,450,04:36,436,07:00,700,...,47.5,81.1,11,Partly cloudy,0,90,9,1012,57,0
2,30-01-2020,DEL,HYD,Air Asia,06:05,605,05:54,554,08:15,815,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,0
3,30-01-2020,DEL,HYD,Air Asia,17:40,1740,17:40,1740,19:55,1955,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,0
4,30-01-2020,DEL,HYD,Air Asia,19:50,1950,19:54,1954,22:10,2210,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10713,24-01-2019,CCU,DEL,SpiceJet,18:30,1830,18:30,1830,20:55,2055,...,13.3,69.2,6,Clear,0,29,10,1016,1,0
10714,23-01-2019,CCU,DEL,SpiceJet,18:30,1830,18:30,1830,20:55,2055,...,13.3,69.2,3,Clear,0,27,10,1015,0,0
10715,22-01-2019,CCU,DEL,SpiceJet,18:30,1830,18:38,1838,20:55,2055,...,13.3,69.2,5,Clear,0,28,10,1015,0,1
10716,09-01-2019,DEL,HYD,Spicejet,22:05,2205,00:40,40,00:25,25,...,13.3,69.2,8,Patchy rain possible,1,55,9,997,85,4


In [4]:
print(len(data["From"].unique()))
#only four different airports it comes from

4


#Adds airport congestion with COUNT GROUPBY using airport 

data["Airport_Congestion"] = (
    data.groupby(["Used Date", "From", "Scheduled Departure"])["Airline"]
    .transform("count")
)

In [5]:
data["Departure_Hour"] = pd.to_datetime(
    data["Scheduled Departure"], format="%H:%M"
).dt.hour #gets hour

# Count flights per airport, date, and hour
hourly_counts = (
    data.groupby(["From", "Used Date", "Departure_Hour"])
    .size()
    .reset_index(name="Flight_Count")
)

# Calculate average flights per airport and hour
avg_congestion = (
    hourly_counts.groupby(["From", "Departure_Hour"])["Flight_Count"]
    .mean()
    .reset_index(name="Avg_Airport_Congestion")
)

# Merge back into original dataset
data = data.merge(
    avg_congestion,
    on=["From", "Departure_Hour"],
    how="left"
)

In [6]:
data["Status"].value_counts()
data = data[data["Status"] == 1].copy()
data.drop(columns=["Status"], inplace=True)

In [7]:
##TEST## Goal is to add features of weather hours before departure 

airports = {
    "DEL": (28.5562, 77.1000), #manually checked
    "BOM": (19.0896, 72.8656),
    "BLR": (13.1986, 77.7066),
    "CCU": (22.6547, 88.4467)
}

weather_variables = [
    "temperature_2m",
    "wind_speed_10m",
    "precipitation",
    "relative_humidity_2m",
    "visibility",
    "pressure_msl",
    "cloud_cover",
    "weather_code"
]

In [8]:
#uses ERA5 data
data["Scheduled_DateTime"] = pd.to_datetime(
    data["Used Date"].astype(str) + " " +
    data["Scheduled Departure"].astype(str),
    dayfirst=True,
    errors="coerce"
)

data["Weather_Cutoff"] = (
    data["Scheduled_DateTime"] - pd.Timedelta(hours=2)
)

start_date = data["Weather_Cutoff"].min().strftime("%Y-%m-%d")
end_date = data["Weather_Cutoff"].max().strftime("%Y-%m-%d")

weather_frames = []

for airport, (lat, lon) in airports.items():

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(weather_variables),
        "timezone": "Asia/Kolkata",
        "models": "era5"
    }

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=120
    )
    response.raise_for_status()

    weather = pd.DataFrame(response.json()["hourly"])
    weather["From"] = airport

    weather_frames.append(weather)

weather_data = pd.concat(weather_frames, ignore_index=True)
weather_data["time"] = pd.to_datetime(weather_data["time"])

print("Downloaded weather rows:", len(weather_data))

Downloaded weather rows: 38016


In [9]:
data = pd.merge_asof(
    data.sort_values("Weather_Cutoff"),
    weather_data.sort_values("time"),
    left_on="Weather_Cutoff",
    right_on="time",
    by="From",
    direction="backward",
    tolerance=pd.Timedelta(hours=1)
)

In [10]:
# Rename new weather columns without touching original columns
weather_mapping = {
    "wind_speed_10m": "Past_windspeedKmph",
    "precipitation": "Past_precipMM",
    "relative_humidity_2m": "Past_humidity",
    "visibility": "Past_visibility",
    "pressure_msl": "Past_pressure",
    "cloud_cover": "Past_cloudcover",
    "weather_code": "Past_weatherDesc"
}

# Convert visibility from meters to kilometers
data["visibility"] = data["visibility"] / 1000

# Rename new historical weather features
data = data.rename(columns=weather_mapping)

# Optional: keep historical temperature too
data = data.rename(columns={
    "temperature_2m": "Past_temperature"
})

In [11]:
original_cols = [
    col for col in data.columns
    if col.startswith("weather__hourly__")
]

past_cols = [
    col for col in data.columns
    if col.startswith("Past_")
]

print(data[original_cols + past_cols].head())

print("\nMissing historical weather:")
print(data[past_cols].isna().sum())

print("\nTotal flights:", len(data))

   weather__hourly__windspeedKmph weather__hourly__weatherDesc__value  \
0                               8                               Clear   
1                               8                Patchy rain possible   
2                               5                               Clear   
3                               3                               Clear   
4                               6                               Clear   

   weather__hourly__precipMM  weather__hourly__humidity  \
0                          0                         47   
1                          1                         55   
2                          0                         28   
3                          0                         27   
4                          0                         29   

   weather__hourly__visibility  weather__hourly__pressure  \
0                           10                       1018   
1                            9                        997   
2                      

In [12]:
data = data.drop(columns=["Past_visibility"])

In [13]:
#TODO
#https://www.moneycontrol.com/news/business/analysis-the-busiest-hour-for-air-travel-in-india-is-not-what-you-would-expect-9100311.html
#Add binary peak hours

In [14]:
#from website

delhi_departures = {
    0: 7,
    1: 7,
    2: 20,
    3: 5,
    4: 50,
    5: 248,   # Corrected: above 240
    6: 240,   # Corrected
    7: 205,
    8: 168,
    9: 195,
    10: 195,
    11: 160,
    12: 130,
    13: 193,
    14: 152,
    15: 176,
    16: 107,
    17: 236,
    18: 206,
    19: 145,
    20: 190,
    21: 117,
    22: 56,
    23: 17
}

data.loc[
    data["From"] == "DEL",
    "Hourly_Airport_Departures"
] = (
    data.loc[data["From"] == "DEL", "Departure_Hour"]
    .map(delhi_departures)
)

In [15]:
mumbai_departures = {
    0: 22,
    1: 7,
    2: 6,
    3: 7,
    4: 22,
    5: 109,
    6: 155,
    7: 154,
    8: 86,
    9: 126,
    10: 108,
    11: 159,
    12: 114,
    13: 105,
    14: 79,
    15: 124,
    16: 110,
    17: 172,
    18: 110,
    19: 131,
    20: 89,
    21: 91,
    22: 77,
    23: 21
}

data.loc[
    data["From"] == "BOM",
    "Hourly_Airport_Departures"
] = (
    data.loc[data["From"] == "BOM", "Departure_Hour"]
    .map(mumbai_departures)
)

In [16]:
bengaluru_departures = {
    0: 8,
    1: 7,
    2: 7,
    3: 7,
    4: 25,
    5: 123,
    6: 154,
    7: 119,
    8: 87,
    9: 120,
    10: 85,
    11: 117,
    12: 80,
    13: 81,
    14: 66,
    15: 110,
    16: 87,
    17: 102,
    18: 105,
    19: 110,
    20: 72,
    21: 78,
    22: 50,
    23: 36
}

data.loc[
    data["From"] == "BLR",
    "Hourly_Airport_Departures"
] = (
    data.loc[data["From"] == "BLR", "Departure_Hour"]
    .map(bengaluru_departures)
)

In [17]:
kolkata_departures = {
    4: 36,
    5: 73,
    6: 59,
    7: 52,
    8: 73,
    9: 72,
    10: 71,
    11: 62,
    12: 55,
    13: 50,
    14: 66,
    15: 66,
    16: 52,
    17: 69,
    18: 90,
    19: 44,
    20: 31,
    21: 52,
    22: 3,
    23: 24
}

data.loc[
    data["From"] == "CCU",
    "Hourly_Airport_Departures"
] = (
    data.loc[data["From"] == "CCU", "Departure_Hour"]
    .map(kolkata_departures)
)

In [18]:
# Maximum hourly departures for each airport
airport_max = {
    "DEL": max(delhi_departures.values()),
    "BOM": max(mumbai_departures.values()),
    "BLR": max(bengaluru_departures.values()),
    "CCU": max(kolkata_departures.values())
}

# Create numerical congestion index
data["Airport_Congestion_Index"] = (
    data["Hourly_Airport_Departures"]
    / data["From"].map(airport_max)
)

In [19]:
data["Used Date"] = pd.to_datetime(
    data["Used Date"], dayfirst=True
) #get DOW

In [20]:
data["Is_Weekend"] = (
    data["Used Date"].dt.dayofweek >= 4
).astype(int)

In [21]:
data["Is_Delayed"] = (data["Departure Delay"] > 0).astype(int)

In [22]:
data["Is_Delayed"].value_counts() #roughly 50/50

Is_Delayed
0    5471
1    5171
Name: count, dtype: int64

In [23]:
print((data["Airline"].unique()))


['Vistara' 'Spicejet' 'SpiceJet' 'Air Asia' 'Indigo' 'Go Air' 'Air India']


In [24]:
data[["Category", "Departure Delay"]].head(20)

,Category,Departure Delay
0,2,20
1,4,155
2,1,8
3,0,0
4,0,0
5,0,0
6,1,8
7,0,-3
8,1,5
9,1,13


In [25]:
columns_to_drop = [
    "Arrival",
    "Scheduled Arrival",
    "Arrival Delay",
    "Departure",
]

data = data.drop(columns=columns_to_drop, errors="ignore")

In [26]:
print(data.columns)

Index(['Used Date', 'From', 'To', 'Airline', 'Scheduled Departure', 'SDEP',
       'DEP', 'SARR', 'ARR', 'Departure Delay', 'Distance',
       'Passenger Load Factor', 'Airline Rating', 'Airport Rating',
       'Market Share', 'OTP Index', 'weather__hourly__windspeedKmph',
       'weather__hourly__weatherDesc__value', 'weather__hourly__precipMM',
       'weather__hourly__humidity', 'weather__hourly__visibility',
       'weather__hourly__pressure', 'weather__hourly__cloudcover', 'Category',
       'Departure_Hour', 'Scheduled_DateTime', 'Weather_Cutoff', 'time',
       'Past_temperature', 'Past_windspeedKmph', 'Past_precipMM',
       'Past_humidity', 'Past_pressure', 'Past_cloudcover', 'Past_weatherDesc',
       'Hourly_Airport_Departures', 'Airport_Congestion_Index', 'Is_Weekend',
       'Is_Delayed'],
      dtype='object')


In [28]:

data = pd.read_csv("flight_data_cleaned.csv")

# Preserve the original data
updated = data.copy()

# Prepare scheduled departure timestamps
updated["Scheduled_DateTime"] = pd.to_datetime(
    updated["Scheduled_DateTime"]
)

# Prepare ERA5 weather data
era5 = weather_data.copy()
era5["time"] = pd.to_datetime(era5["time"])

era5_cols = [
    "temperature_2m",
    "wind_speed_10m",
    "precipitation",
    "relative_humidity_2m",
    "pressure_msl",
    "cloud_cover",
    "weather_code"
]

# Keep a flight identifier to preserve row order
updated["_row_id"] = range(len(updated))

# Remove the old ERA5 merge timestamp
updated = updated.drop(columns=["time"], errors="ignore")

# Match each flight to nearest ERA5 hourly observation
updated = pd.merge_asof(
    updated.sort_values("Scheduled_DateTime"),
    era5[["From", "time"] + era5_cols].sort_values("time"),
    left_on="Scheduled_DateTime",
    right_on="time",
    by="From",
    direction="nearest",
    tolerance=pd.Timedelta("30min")
)

updated = updated.sort_values("_row_id").drop(columns="_row_id")

In [29]:

import pandas as pd

# Load original flight data
data = pd.read_csv("flight_data_cleaned.csv")
updated = data.copy()

# Remove old hourly weather columns
old_weather_cols = [
    col for col in updated.columns
    if col.startswith("weather__hourly__")
]

updated = updated.drop(columns=old_weather_cols)

# Convert scheduled departure from IST to UTC
updated["Scheduled_DateTime"] = (
    pd.to_datetime(updated["Scheduled_DateTime"])
    .dt.tz_localize("Asia/Kolkata")
    .dt.tz_convert("UTC")
)

# Prepare ERA5 weather data (assumed UTC)
era5 = weather_data.copy()
era5["time"] = pd.to_datetime(era5["time"], utc=True)

era5_cols = [
    "temperature_2m",
    "wind_speed_10m",
    "precipitation",
    "relative_humidity_2m",
    "pressure_msl",
    "cloud_cover",
    "weather_code"
]

# Preserve original row order
updated["_row_id"] = range(len(updated))

# Remove any previous merge columns
updated = updated.drop(
    columns=["time"] + era5_cols,
    errors="ignore"
)

# Ensure matching airport codes are consistent
updated["From"] = updated["From"].astype(str).str.strip()
era5["From"] = era5["From"].astype(str).str.strip()

# Match nearest ERA5 hourly weather observation
updated = pd.merge_asof(
    updated.sort_values("Scheduled_DateTime"),
    era5[["From", "time"] + era5_cols].sort_values("time"),
    left_on="Scheduled_DateTime",
    right_on="time",
    by="From",
    direction="nearest",
    tolerance=pd.Timedelta("30min")
)

# Restore original flight order
updated = updated.sort_values("_row_id").drop(
    columns=["_row_id", "time"]
)

# Convert scheduled departure back to IST
updated["Scheduled_DateTime"] = (
    updated["Scheduled_DateTime"]
    .dt.tz_convert("Asia/Kolkata")
    .dt.tz_localize(None)
)

# Check results
print("Original rows:", len(data))
print("Updated rows:", len(updated))
print(
    "ERA5 matching rate:",
    round(updated["temperature_2m"].notna().mean() * 100, 2),
    "%"
)

# Save new dataset
updated.to_csv("flight_data_ERA5.csv", index=False)


Original rows: 10642
Updated rows: 10642
ERA5 matching rate: 100.0 %


In [ ]:
weather_mapping = {
    "weather__hourly__windspeedKmph": "wind_speed_10m",
    "weather__hourly__precipMM": "precipitation",
    "weather__hourly__humidity": "relative_humidity_2m",
    "weather__hourly__pressure": "pressure_msl",
    "weather__hourly__cloudcover": "cloud_cover",
    "weather__hourly__weatherDesc__value": "weather_code"
}

for original, replacement in weather_mapping.items():
    updated[original] = updated[replacement]

# ERA5 temperature has no corresponding original column
updated["ERA5_Scheduled_Temperature"] = updated["temperature_2m"]

# ERA5 visibility wasn't available in your earlier data
updated = updated.drop(
    columns=["weather__hourly__visibility"],
    errors="ignore"
)

# Remove temporary ERA5 columns
updated = updated.drop(
    columns=era5_cols + ["time"],
    errors="ignore"
)

print("Original flights:", len(data))
print("Updated flights:", len(updated))
print("Missing scheduled ERA5 wind:", updated[
    "weather__hourly__windspeedKmph"
].isna().sum())

updated.to_csv("flight_data_ERA5_scheduled.csv", index=False)

Original flights: 10642
Updated flights: 10642
Missing scheduled ERA5 wind: 0


In [ ]:
data["Is_Delayed"] = (data["Departure Delay"] > 0).astype(int)

print(data["Is_Delayed"].value_counts(normalize=True) * 100)

Is_Delayed
0    51.409509
1    48.590491
Name: proportion, dtype: float64
